<a href="https://colab.research.google.com/github/diegocortes30/olist_ecommerce_analysis/blob/main/notebooks_01_data_understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Olist E-commerce Analysis: 01 Data Understanding**

**Goal:** understand the structure and quality of the Olist Brazilian e-commerce dataset before analysing it.

**Data:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle), 9 CSV files.

**Contents:** setup, types and missing values, checks between tables, data quality findings, data dictionary, table relationships.

## 1. Setup and data loading

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import os
RAW = '/content/drive/MyDrive/Olist/raw'
os.listdir(RAW)

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [33]:
import pandas as pd

tables = {}
for file in os.listdir(RAW):
  if file.endswith('.csv'):
    name = file.replace('olist_', '').replace('_dataset.csv', '').replace('.csv', '')
    tables[name] = pd.read_csv(f'{RAW}/{file}')

for name, df in tables.items():
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

customers: 99,441 rows, 5 columns
geolocation: 1,000,163 rows, 5 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
orders: 99,441 rows, 8 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
product_category_name_translation: 71 rows, 2 columns


## 2. Column types and missing values

In [34]:
for name, df in tables.items():
  print(f'\n--- {name} ---')
  print(df.dtypes)


--- customers ---
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

--- geolocation ---
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

--- order_items ---
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

--- order_payments ---
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

--- order_reviews ---
review_id                  object
order_id                   object
review_score         

In [35]:
# Count null values
for name, df in tables.items():
  nulls = df.isnull().sum()
  nulls = nulls[nulls > 0]
  if len(nulls) > 0:
    print(f'\n--- {name} ---')
    print(nulls)


--- order_reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [36]:
# Verify duplicates in orders and order_reviews
tables['order_reviews']['order_id'].duplicated().sum()

np.int64(551)

## 3. Missing delivery dates

In [37]:
# Hypothesis: nulls come from orders that were never delivered
orders = tables['orders']
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [38]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

,count
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


## 4. Orders with several reviews

In [39]:
reviews = tables['order_reviews']
dups = reviews[reviews['order_id'].duplicated(keep=False)]
dups.sort_values('order_id').head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06


In [40]:
reviews['order_id'].value_counts().value_counts()

,count
count,
1,98126
2,543
3,4


## 5. Orders without items

In [41]:
set(tables['order_items']['order_id']).issubset(set(tables['orders']['order_id']))

True

In [42]:
set(tables['orders']['order_id']).issubset(set(tables['order_items']['order_id']))

False

In [43]:
no_items = ~orders['order_id'].isin(tables['order_items']['order_id'])
no_items.sum()

np.int64(775)

In [44]:
orders[no_items]['order_status'].value_counts()

,count
order_status,
unavailable,603
canceled,164
created,5
invoiced,2
shipped,1


# **Data quality findings**

1. **Date columns are stored as text:** All date columns in `orders`, `order_items`
   and `order_reviews` have the type `object` instead of `datetime`.
   *Action:* convert them with `pd.to_datetime` during cleaning.

2. **Missing dates in `orders`:** There are 2,965 orders without a delivery date.
   Most have a status such as `shipped`, `canceled` or `unavailable`, so the nulls
   reflect orders that were never delivered, not data errors.
   *Action:* keep the nulls and do not impute them.

3. **Inconsistent delivered orders:** Eight orders have the status `delivered`
   but no delivery date.
   *Action:* exclude them from delivery-time analysis.

4. **Orders with multiple reviews:** 547 orders have more than one review
   (551 extra rows), which would duplicate rows when joining tables.
   *Action:* keep one review per order.

5. **Orders without items:** 775 orders have no rows in `order_items`; 99.6% are
   `unavailable`, `canceled` or `created`.
   *Action:* exclude them from sales analysis.

6. **Zip code prefixes lost their leading zeros.** `customer_zip_code_prefix` is stored as an integer, so a code like 01003 appears as 1003.
   *Action:* convert it to text padded to 5 digits (`zfill(5)`) before joining with `geolocation`.

7. **Products with missing attributes.** 610 products have no category, name, description or photos.
   *Action:* keep them and label the category as "unknown".

8. **Categories without translation.** 2 categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) are missing from the translation table.
   *Action:* add the translations manually during cleaning.

9. **Geolocation has many rows per zip code.** 19,015 distinct zip prefixes across 1,000,163 rows, and 261,831 rows are exact duplicates. Joining it directly would multiply the rows of customers or sellers.
   *Action:* remove duplicates and average latitude and longitude per zip prefix before joining.
   
10. **A review can be linked to several orders.** 814 rows repeat a `review_id`
    with a different `order_id`, so `review_id` is not a primary key.
    *Action:* count distinct `review_id` when counting reviews, and use one row per order for order-level analysis.


## **Data dictionary**

### orders (99,441 rows, one row per order)

| Column | Type | Description |
|---|---|---|
| order_id | text | Unique identifier of the order (primary key) |
| customer_id | text | Identifier of the customer for this order; a new one is assigned per order (links to `customers`) |
| order_status | text | Status of the order (delivered, shipped, canceled, etc.) |
| order_purchase_timestamp | datetime |Date and time when the order was placed |
| order_approved_at | datetime |Date and time when the order was approved |
| order_delivered_carrier_date | datetime |Date when the order was handed to the logistics carrier |
| order_delivered_customer_date | datetime |Date when the order actually reached the customer
| order_estimated_delivery_date | datetime |Delivery date promised to the customer at purchase |



### order_items (112,650 rows, one row per item in an order)

| Column | Type | Description |
|---|---|---|
| order_id | text | Identifier of the order (links to `orders`) |
| order_item_id | integer |Sequence number of the item within its order (1, 2, 3...); together with `order_id` it identifies each row |
| product_id | text |Identifier of the product (links to `products`) |
| seller_id | text |Identifier of the seller (links to `sellers`) |
| shipping_limit_date | datetime |Deadline for the seller to hand the item to the carrier |
| price | float | Price of a single item |
| freight_value | float |Shipping cost allocated to the item  |


### customers (99,441 rows, one row per order's customer record)

| Column | Type | Description |
|---|---|---|
| customer_id | text | Identifier assigned to the customer for a given order (links to `orders`) |
| customer_unique_id | text |Identifier of the actual person; stays the same across orders, so it is used to count repeat customers (96,096 distinct values)  |
| customer_zip_code_prefix | integer |First digits of the customer's zip code |
| customer_city | text |City where the customer lives |
| customer_state | text |State where the customer lives (two-letter code) |


### order_payments (103,886 rows, one row per payment of an order)

| Column | Type | Description |
|---|---|---|
| order_id | text | Identifier of the order (links to `orders`) |
| payment_sequential | integer |Sequence number of the payment within its order (1, 2, 3...); an order can be paid with several payments |
| payment_type | text |Payment method: `credit_card`, `boleto`, `voucher`, `debit_card` or `not_defined` |
| payment_installments | integer |Number of installments chosen by the customer; 1 means paid in a single payment |
| payment_value | float |Value of this payment|

### order_reviews (99,224 rows, one row per review-order pair)

| Column | Type | Description |
|---|---|---|
| review_id | text |Identifier of the review; not unique, since the same review can be linked to several orders (814 repeated rows) |
| order_id | text | Identifier of the order (links to `orders`); some orders have more than one review |
| review_score | integer |Customer satisfaction rating of the order, from 1 (worst) to 5 (best)  |
| review_comment_title | text |Optional title of the comment (empty in 87,656 rows)|
| review_comment_message | text |Optional comment written by the customer (empty in 58,247 rows) |
| review_creation_date | datetime |Date when the satisfaction survey was sent to the customer |
| review_answer_timestamp | datetime |Date and time when the customer answered the survey |


### products (32,951 rows, one row per product)

| Column | Type | Description |
|---|---|---|
| product_id | text | Identifier of the product (links to `order_items`) |
| product_category_name | text |Category of the product, in Portuguese; empty in 610 rows |
| product_name_lenght | float |Number of characters in the product's name (column name misspelled in the source); empty in 610 rows  |
| product_description_lenght | float |Number of characters in the product's description (misspelled in the source); empty in 610 rows|
| product_photos_qty | float |Number of photos of the product  |
| product_weight_g | float |Weight of the product (g) |
| product_length_cm | float | Lenght of the product (cm)|
| product_height_cm | float |Height of the product (cm) |
| product_width_cm | float |Widht of the product (cm) |


### sellers (3,095 rows, one row per seller)

| Column | Type | Description |
|---|---|---|
| seller_id | text |Identifier of the seller (primary key; links to `order_items`)  |
| seller_zip_code_prefix | integer |First digits of the seller's zip code (an area, not an exact address) |
| seller_city | text |City where the seller is located |
| seller_state | text |State where the seller is located (two-letter code) |

### product_category_name_translation (71 rows, one row per category)

| Column | Type | Description |
|---|---|---|
| product_category_name | text |Name of the category in Portuguese (primary key; links to `products`) |
| product_category_name_english | text |Name of the category in English |


### geolocation (1,000,163 rows, one row per coordinate record)

| Column | Type | Description |
|---|---|---|
| geolocation_zip_code_prefix | integer | First digits of a Brazilian zip code (an area); repeated many times, with 19,015 distinct values across the table (links to `customers` and `sellers`) |
| geolocation_lat | float |Latitude of a coordinate record inside the zip code area |
| geolocation_lng | float |Longitude of a coordinate record inside the zip code area |
| geolocation_city | text |City of the record |
| geolocation_state | text |State of the record |


## Table relationships

| From table | Column | To table | Relationship |
|---|---|---|---|
| orders | customer_id | customers | one-to-one: each order has its own customer record |
| order_items | order_id | orders | many-to-one: an order can have up to 21 items |
| order_items | product_id | products | many-to-one: a product can appear in many items |
| order_items | seller_id | sellers | many-to-one: a seller can sell many items |
| order_payments | order_id | orders | many-to-one: an order can have several payments |
| order_reviews | order_id | orders | many-to-one: 547 orders have more than one review |
| products | product_category_name | product_category_name_translation | many-to-one: many products share a category; 2 categories have no translation |
| customers / sellers | zip_code_prefix | geolocation | one-to-many: a zip prefix has many coordinate records; reduce `geolocation` before joining |

## Evidence for the data dictionary

In [45]:
products = tables['products']
translation = tables['product_category_name_translation']
payments = tables['order_payments']
reviews = tables['order_reviews']
geo = tables['geolocation']

print('orders.order_id unique:', orders['order_id'].is_unique)
print('products.product_id unique:', products['product_id'].is_unique)
print('sellers.seller_id unique:', tables['sellers']['seller_id'].is_unique)
print('customer_id distinct:', tables['customers']['customer_id'].nunique())
print('customer_unique_id distinct:', tables['customers']['customer_unique_id'].nunique())
print('min customer zip prefix:', tables['customers']['customer_zip_code_prefix'].min())
print('order_items key duplicates:', tables['order_items'].duplicated(subset=['order_id', 'order_item_id']).sum())
print('payments key duplicates:', payments.duplicated(subset=['order_id', 'payment_sequential']).sum())
print('orders without payments:', (~orders['order_id'].isin(payments['order_id'])).sum())
print('review_id unique:', reviews['review_id'].is_unique)
print('geo distinct zip prefixes:', geo['geolocation_zip_code_prefix'].nunique())
print('geo duplicated rows:', geo.duplicated().sum())
print('categories without translation:',
      set(products['product_category_name'].dropna()) - set(translation['product_category_name']))
reviews['review_score'].value_counts().sort_index()

orders.order_id unique: True
products.product_id unique: True
sellers.seller_id unique: True
customer_id distinct: 99441
customer_unique_id distinct: 96096
min customer zip prefix: 1003
order_items key duplicates: 0
payments key duplicates: 0
orders without payments: 1
review_id unique: False
geo distinct zip prefixes: 19015
geo duplicated rows: 261831
categories without translation: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


,count
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328
